In [26]:
#                    #
##                  ##
######################
### Build datacard ###
######################
######################

import numpy as np
import pandas as pd
import json
import os
import sys
import functools
import json

def getZhbbWeight(df_, year):
    tot_weight = (df_['norm_weight'] * np.sign(df_['genWeight']) * df_['topptWeight'])# * 
                  #df_['ele_reco_sf'] * df_['ele_id_sf'] * df_['mu_id_sf'] * df_['mu_iso_sf'] *
                  #df_['bbtag_sf'] * df_['btag_sf'] * df_['puWeight']) # and other weights
    return tot_weight
pt_bins = [0,200,300,450, np.inf]
#pt_bins = [200,300,450]
mass_bins = [50,80,105,145,200]
#mass_bins = [50,80,105,145]
isblind = True

In [27]:
def cuts(df_):
    base = (
    (df_['n_ak4jets']   >= 5)       &
    (df_['n_b_outZH'] == 2) &
    (df_['ZH_bbvLscore'] >= 0.9105) &
    #(df_['ZH_pt']       >= 200)& # 200
    (df_['MET_pt']      >= 20)            &
    (df_['ZH_M']        >= 50)            &
    (df_['ZH_M']        <= 200)
    )
    return base

def stxs_cut(df_):
    base = (
        (df_['ZH_rap'] <= 2.5)
    )
    return base

In [28]:
def calculate_systematics(target_df, nominal_stxs_yield, year, sample_name):
    sys_results = {}
    systematics = [
        'isr_up', 'isr_down', 'fsr_up', 'fsr_down', 
        'mu_r_up', 'mu_r_down', 'mu_f_up', 'mu_f_down'
    ]
    
    # Pre-filter the dataframe to save computation time
    mask = (target_df['process'] == sample_name) & stxs_cut(target_df)
    filtered_df = target_df[mask]
    
    for sys in systematics:
        # Only calculate if the column actually exists in the dataframe
        if sys in target_df.columns:
            if nominal_stxs_yield != 0:
                # Calculate the varied yield (denominator)
                denom = np.sum(getZhbbWeight(filtered_df, year) * filtered_df[sys])
                # Safely divide, defaulting to 1.0 if denom is 0
                sys_results[sys] = (nominal_stxs_yield / denom) if denom != 0 else 1.0
            else:
                sys_results[sys] = 1.0
                
    return sys_results

In [29]:
samples = ['ttH', 'ttZ', 'TTBar', 'tt_B']#, 'QCD', 'VJets']
get_pickle= (lambda s: pd.read_pickle(f'pickled/Inference_{s}.pkl'))
df = pd.concat([get_pickle(s) for s in samples], axis='rows', ignore_index=True)
#df = df[cuts(df)]
outer_dict = {}
inner_dict = {}

pt_bins = [0, 200, 300, 450, np.inf]

for y in ['2024']:
    for sample in samples:
        _df = df[cuts(df)].copy() 
        _df['pt_bin'] = pd.cut(
            _df['genZHpt'], 
            bins=pt_bins, 
            labels=[f'Zhpt{i}' for i in range(len(pt_bins) - 1)]
        )
        
        # Calculate inclusive yields
        sumw = np.sum(getZhbbWeight(_df[_df['process'] == sample], y))
        sumw_stxs = np.sum(getZhbbWeight(_df[(_df['process'] == sample) & stxs_cut(_df)], y))
        
        process_dict = {
            "yield" : sumw,
            "stxs_yield": sumw_stxs,
        }
        
        # --- NEW: Dynamically add systematics to inclusive dict ---
        sys_dict = calculate_systematics(_df, sumw_stxs, y, sample)
        process_dict.update(sys_dict) 
        
        if sample == 'tt_B':
            inner_dict['ttbb'] = process_dict
        else:
            inner_dict[sample] = process_dict
            
        # ---------------------------------------------------------
        # Differential Yields (ttH and ttZ only)
        # ---------------------------------------------------------
        if sample in ['ttH', 'ttZ']:
            for i in range(len(pt_bins) - 1):
                bin_df = _df[_df['pt_bin'] == f'Zhpt{i}']
                
                bin_sumw = np.sum(getZhbbWeight(bin_df[bin_df['process'] == sample], y))
                bin_sumw_stxs = np.sum(getZhbbWeight(bin_df[(bin_df['process'] == sample) & stxs_cut(bin_df)], y))
                
                bin_process_dict = {
                    "yield" : bin_sumw,
                    "stxs_yield": bin_sumw_stxs,
                }
                
                # --- NEW: Dynamically add systematics to differential dict ---
                bin_sys_dict = calculate_systematics(bin_df, bin_sumw_stxs, y, sample)
                bin_process_dict.update(bin_sys_dict)
                
                inner_dict[f'{sample}{i}'] = bin_process_dict

    outer_dict[y] = inner_dict

In [30]:
outer_dict

{'2024': {'ttH': {'yield': 37.32039394298088, 'stxs_yield': 37.32039394298088},
  'ttH0': {'yield': 8.244748406310956, 'stxs_yield': 8.244748406310956},
  'ttH1': {'yield': 5.0482153848193185, 'stxs_yield': 5.0482153848193185},
  'ttH2': {'yield': 16.14627643328061, 'stxs_yield': 16.14627643328061},
  'ttH3': {'yield': 7.881153718569985, 'stxs_yield': 7.881153718569985},
  'ttZ': {'yield': 19.569507095245704, 'stxs_yield': 19.569507095245704},
  'ttZ0': {'yield': 1.8565942628822847, 'stxs_yield': 1.8565942628822847},
  'ttZ1': {'yield': 6.707155850592759, 'stxs_yield': 6.707155850592759},
  'ttZ2': {'yield': 7.376198828207997, 'stxs_yield': 7.376198828207997},
  'ttZ3': {'yield': 3.629558153562665, 'stxs_yield': 3.629558153562665},
  'TTBar': {'yield': 96.31748150602571, 'stxs_yield': 96.31748150602571},
  'ttbb': {'yield': 46.58717201725942, 'stxs_yield': 46.58717201725942}}}

In [31]:
out_json_file = './process_norms/process_norms_run3.json'
with open(out_json_file, 'w') as jsf:
    json.dump(outer_dict, jsf, indent=4)

In [ ]:
df = df[cuts(df)]

In [ ]:
sumw = np.sum(getZhbbWeight(df, '2017'))

In [ ]:
process_dict = {
    "yield" : sumw
}